# Notebook Nro 8 - variantes en etiqueta persona

# Celda 1 — Instalar dependencia para similitud

Vamos a usar rapidfuzz, que sirve para comparar textos parecidos.

In [1]:
!pip install -q rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 30.7 MB/s eta 0:00:00



## Celda 2 — Imports y explicación

In [2]:
# =========================
# NOTEBOOK 8
# Agrupar variantes de entidad persona
# =========================
#
# Objetivo:
# Crear un JSON/CSV donde las entidades persona estén agrupadas por persona probable.
#
# Ejemplo:
# PETRONE Maria Teresa
# María Taresa Petrone
# PETRONE Marta Teresa
#
# quedan dentro del mismo grupo de variantes.

import json
import re
import unicodedata
from pathlib import Path

import pandas as pd
from rapidfuzz import fuzz

# Celda 3 — Subir archivos

In [3]:
from google.colab import files

uploaded = files.upload()

print("Archivos subidos:")
for filename in uploaded.keys():
    print("-", filename)

Saving embargos_revision_entidades_actualizado_metodo_corregido.csv to embargos_revision_entidades_actualizado_metodo_corregido.csv
Saving embargos_revision_entidades_actualizado_metodo_corregido.json to embargos_revision_entidades_actualizado_metodo_corregido.json
Archivos subidos:
- embargos_revision_entidades_actualizado_metodo_corregido.csv
- embargos_revision_entidades_actualizado_metodo_corregido.json


# Celda 4 — Detectar archivos automáticamente

In [4]:
uploaded_files = list(uploaded.keys())

json_files = [f for f in uploaded_files if f.lower().endswith(".json")]
csv_files = [f for f in uploaded_files if f.lower().endswith(".csv")]

if len(json_files) != 1:
    raise ValueError(f"Se esperaba 1 JSON, encontrados: {json_files}")

if len(csv_files) != 1:
    raise ValueError(f"Se esperaba 1 CSV, encontrados: {csv_files}")

JSON_PATH = Path(json_files[0])
CSV_PATH = Path(csv_files[0])

print("JSON detectado:", JSON_PATH)
print("CSV detectado:", CSV_PATH)

JSON detectado: embargos_revision_entidades_actualizado_metodo_corregido.json
CSV detectado: embargos_revision_entidades_actualizado_metodo_corregido.csv


# Celda 5 — Configurar salidas

In [5]:
OUTPUT_JSON = Path("/content/embargos_personas_agrupadas_variantes.json")
OUTPUT_CSV = Path("/content/embargos_personas_agrupadas_variantes.csv")
OUTPUT_EXCEL = Path("/content/embargos_personas_agrupadas_variantes.xlsx")

print("Salida JSON:", OUTPUT_JSON)
print("Salida CSV:", OUTPUT_CSV)
print("Salida Excel:", OUTPUT_EXCEL)

Salida JSON: /content/embargos_personas_agrupadas_variantes.json
Salida CSV: /content/embargos_personas_agrupadas_variantes.csv
Salida Excel: /content/embargos_personas_agrupadas_variantes.xlsx


# Celda 5 — Cargar JSON y CSV

In [6]:
with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.read_csv(CSV_PATH)

print("Documentos en JSON:", len(data))
print("Filas en CSV:", len(df))
print("Columnas CSV:")
print(df.columns.tolist())

Documentos en JSON: 80
Filas en CSV: 1626
Columnas CSV:
['numero_archivo', 'id', 'nombre_archivo', 'clasificacion', 'texto_limpio', 'cantidad_entidades_encontradas', 'etiqueta', 'valor', 'metodo', 'span_inicio', 'span_fin']


# Celda 6 — Cargar datos

In [7]:
with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.read_csv(CSV_PATH)

print("Documentos cargados:", len(data))
print("Filas CSV:", len(df))

Documentos cargados: 80
Filas CSV: 1626


# Celda 7 — Funciones de normalización

In [8]:
def clean_value(x):
    if x is None:
        return None

    try:
        if pd.isna(x):
            return None
    except Exception:
        pass

    s = str(x).strip()
    return s if s else None


def clean_int_or_none(x):
    if x is None:
        return None

    try:
        if pd.isna(x):
            return None
    except Exception:
        pass

    try:
        return int(float(x))
    except Exception:
        return None


def quitar_tildes(texto):
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(ch for ch in texto if unicodedata.category(ch) != "Mn")
    return texto


def normalizar_nombre_persona(nombre):
    """
    Normaliza nombres para comparar variantes.
    No modifica el valor final guardado, solo se usa para agrupar.
    """
    nombre = clean_value(nombre)

    if nombre is None:
        return ""

    nombre = quitar_tildes(nombre)
    nombre = nombre.upper()

    # Quitar títulos y roles frecuentes
    nombre = re.sub(r"\b(DR|DRA|SR|SRA|SEÑOR|SEÑORA|JUEZ|JUEZA|SECRETARIO|SECRETARIA)\b\.?", " ", nombre)

    # Quitar símbolos
    nombre = re.sub(r"[^A-ZÑ\s]", " ", nombre)

    # Espacios múltiples
    nombre = re.sub(r"\s+", " ", nombre).strip()

    return nombre


def tokens_nombre(nombre):
    normalizado = normalizar_nombre_persona(nombre)

    tokens = [
        t for t in normalizado.split()
        if len(t) >= 2
    ]

    return tokens


def firma_tokens(nombre):
    """
    Firma ordenada por tokens.
    Sirve para detectar:
    - JUAN PEREZ
    - PEREZ JUAN
    """
    tokens = tokens_nombre(nombre)
    return " ".join(sorted(tokens))

# Celda 8 — Función para decidir si dos nombres son variantes

In [9]:
def son_variantes_misma_persona(nombre_a, nombre_b):
    """
    Decide si dos nombres probablemente representan la misma persona.

    Cubre casos:
    - mismo nombre exacto normalizado
    - mismo nombre en distinto orden
    - nombre completo vs nombre parcial
    - pequeñas diferencias OCR
    """
    norm_a = normalizar_nombre_persona(nombre_a)
    norm_b = normalizar_nombre_persona(nombre_b)

    if not norm_a or not norm_b:
        return False

    if norm_a == norm_b:
        return True

    tokens_a = tokens_nombre(nombre_a)
    tokens_b = tokens_nombre(nombre_b)

    if not tokens_a or not tokens_b:
        return False

    set_a = set(tokens_a)
    set_b = set(tokens_b)

    # Mismas palabras en distinto orden
    if set_a == set_b:
        return True

    # Si uno está contenido en otro y tiene al menos 2 tokens
    menor = set_a if len(set_a) <= len(set_b) else set_b
    mayor = set_b if len(set_a) <= len(set_b) else set_a

    if len(menor) >= 2 and menor.issubset(mayor):
        return True

    # Caso de una sola palabra, por ejemplo solo apellido.
    # Lo dejamos más estricto para no mezclar personas por error.
    if len(menor) == 1:
        token = list(menor)[0]

        if len(token) >= 5 and token in mayor:
            # Solo se acepta si además la similitud parcial es alta
            if fuzz.partial_ratio(norm_a, norm_b) >= 88:
                return True

    # Similitud por orden flexible
    token_sort = fuzz.token_sort_ratio(norm_a, norm_b)
    token_set = fuzz.token_set_ratio(norm_a, norm_b)

    # Para OCR: MARIA TARESA PETRONE vs PETRONE MARTA TERESA
    if token_sort >= 82 or token_set >= 88:
        return True

    return False

# Celda 9 — Agrupar personas dentro de cada archivo

Esta es la celda importante.

In [10]:
def agrupar_variantes_persona(personas):
    """
    Agrupa entidades persona de un mismo archivo en grupos de variantes.

    Cada grupo representa una persona probable.
    Cada variante representa una aparición o forma distinta detectada.
    """
    grupos = []

    for persona in personas:
        valor = clean_value(persona.get("valor"))

        if valor is None:
            continue

        agregado = False

        for grupo in grupos:
            # Comparamos contra todas las variantes existentes del grupo
            if any(
                son_variantes_misma_persona(valor, variante["valor"])
                for variante in grupo
            ):
                grupo.append(persona)
                agregado = True
                break

        if not agregado:
            grupos.append([persona])

    return grupos

# Celda 10 — Crear JSON correcto con grupos de variantes

In [11]:
json_personas_agrupadas = []

for doc in data:
    numero_archivo = int(doc["numero_archivo"])

    entidades_originales = doc.get("entidades", [])

    # Nos quedamos solo con personas
    personas = [
        ent for ent in entidades_originales
        if str(ent.get("etiqueta", "")).strip().lower() == "persona"
        and clean_value(ent.get("valor")) is not None
    ]

    grupos_persona = agrupar_variantes_persona(personas)

    entidades_finales = []

    for idx_grupo, grupo in enumerate(grupos_persona, start=1):
        variantes = []

        for idx_variante, ent in enumerate(grupo, start=1):
            variantes.append({
                "id_variante": str(idx_variante),
                "valor": clean_value(ent.get("valor")),
                "metodo": clean_value(ent.get("metodo")),
                "span_inicio": clean_int_or_none(ent.get("span_inicio")),
                "span_fin": clean_int_or_none(ent.get("span_fin")),
            })

        entidades_finales.append({
            "id_etiqueta": f"{numero_archivo:03d}_persona_{idx_grupo:03d}",
            "etiqueta": "persona",
            "variantes": variantes,
        })

    nuevo_doc = {
        "numero_archivo": numero_archivo,
        "id": doc.get("id"),
        "nombre_archivo": doc.get("nombre_archivo"),
        "clasificacion": doc.get("clasificacion"),
        "texto_limpio": doc.get("texto_limpio"),
        "entidades": entidades_finales,
    }

    json_personas_agrupadas.append(nuevo_doc)

print("Documentos procesados:", len(json_personas_agrupadas))

print("\nEjemplo primer documento:")
print(json.dumps(json_personas_agrupadas[0], ensure_ascii=False, indent=2)[:4000])

Documentos procesados: 80

Ejemplo primer documento:
{
  "numero_archivo": 1,
  "id": "538118",
  "nombre_archivo": "Embargo - usuario",
  "clasificacion": "Embargo",
  "texto_limpio": "18/2/46. 14:59 TEXTO Y DATOS DE LA NOTIFICACIÓN - SUPREMA CORTE DE JUS 5ncas/ y\n\nin | / /\no 000800 000 vo)\n\nUsuafio conectado: THEILER Pablo Maximiliano - 203222152320 notr\n\n| Orgañismo: juzgado EN LO CIVIL Y COMERCIAL N0%16 - SA\n| Chrátula: o AS SA C/ MARQUEZ NA 02 Ln e eu CUTIVO\nNúmero de causa: s/prrse-2093 o\nTipo de notificación: Jeeps ELECTRONICO 4 paar\nDestinatarios: A Y\nFécha notificación: 20/2/2026 \" O.\nAlta a disponibilidad 19/2/2026 10:44:34\n| Fifmal digital: Firma valida\nFirmablo y Notificado por: PETRONE Maria Teresa. JUEZ --- Certificado Correcto. Fecha de Firma: 19/02/2026\n10:44.33\nFifmabo por: PETRONE Maria Teresa. JUEZ --- Certificado Correcto. Fecha de Firma: 19/02/2026\n\n2]¡BRANDARIZ Luciana Rocio. --- Certificado Correcto. Fecha de Firma:\n2:20p6 32:44.42 THEILER Pa

# Celda 11 — Guardar JSON

In [12]:
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(json_personas_agrupadas, f, ensure_ascii=False, indent=2)

print("JSON guardado:", OUTPUT_JSON)

JSON guardado: /content/embargos_personas_agrupadas_variantes.json


# Celda 12 — Crear CSV aplanado

Cada fila es una variante, pero ahora con su grupo/persona probable.

In [13]:
rows = []

for doc in json_personas_agrupadas:
    for entidad in doc["entidades"]:
        for variante in entidad["variantes"]:
            rows.append({
                "numero_archivo": doc["numero_archivo"],
                "id": doc["id"],
                "nombre_archivo": doc["nombre_archivo"],
                "clasificacion": doc["clasificacion"],
                "texto_limpio": doc["texto_limpio"],

                "id_etiqueta": entidad["id_etiqueta"],
                "etiqueta": entidad["etiqueta"],

                "id_variante": variante["id_variante"],
                "valor": variante["valor"],
                "metodo": variante["metodo"],
                "span_inicio": variante["span_inicio"],
                "span_fin": variante["span_fin"],
            })

df_personas_agrupadas = pd.DataFrame(rows)

df_personas_agrupadas.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("CSV guardado:", OUTPUT_CSV)
print("Total variantes:", len(df_personas_agrupadas))
print("Total grupos persona:", df_personas_agrupadas["id_etiqueta"].nunique())

df_personas_agrupadas.head()

CSV guardado: /content/embargos_personas_agrupadas_variantes.csv
Total variantes: 1031
Total grupos persona: 779


,numero_archivo,id,nombre_archivo,clasificacion,texto_limpio,id_etiqueta,etiqueta,id_variante,valor,metodo,span_inicio,span_fin
0,1,538118,Embargo - usuario,Embargo,18/2/46. 14:59 TEXTO Y DATOS DE LA NOTIFICACIÓ...,001_persona_001,persona,1,THEILER Pablo Maximiliano,gliner_fastino,127,152
1,1,538118,Embargo - usuario,Embargo,18/2/46. 14:59 TEXTO Y DATOS DE LA NOTIFICACIÓ...,001_persona_002,persona,1,PETRONE Maria Teresa,gliner_fastino,616,636
2,1,538118,Embargo - usuario,Embargo,18/2/46. 14:59 TEXTO Y DATOS DE LA NOTIFICACIÓ...,001_persona_002,persona,2,María Taresa Petrone,gliner_fastino,1322,1342
3,1,538118,Embargo - usuario,Embargo,18/2/46. 14:59 TEXTO Y DATOS DE LA NOTIFICACIÓ...,001_persona_002,persona,3,PETRONE Maria Teresa,gliner_fastino,3379,3399
4,1,538118,Embargo - usuario,Embargo,18/2/46. 14:59 TEXTO Y DATOS DE LA NOTIFICACIÓ...,001_persona_002,persona,4,PETRONE Marta Teresa,gliner_fastino,5358,5378


# Celda 13 — Excel de revisión

In [14]:
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Border, Side, Alignment
from openpyxl.utils import get_column_letter

df_excel = df_personas_agrupadas.copy()

# texto_limpio solo en la primera fila de cada archivo
if not df_excel.empty:
    df_excel["texto_limpio"] = df_excel.groupby("numero_archivo")["texto_limpio"].transform(
        lambda s: [str(s.iloc[0])[:1200]] + [""] * (len(s) - 1)
    )

df_excel.to_excel(OUTPUT_EXCEL, index=False, sheet_name="Personas_Agrupadas")

wb = load_workbook(OUTPUT_EXCEL)
ws = wb["Personas_Agrupadas"]

header_fill = PatternFill("solid", fgColor="17365D")
header_font = Font(color="FFFFFF", bold=True)
thin = Side(style="thin", color="D9E2F3")
border = Border(left=thin, right=thin, top=thin, bottom=thin)

fill_odd = PatternFill("solid", fgColor="EAF3FF")
fill_even = PatternFill("solid", fgColor="FFF2CC")
fill_group_odd = PatternFill("solid", fgColor="9DC3E6")
fill_group_even = PatternFill("solid", fgColor="FFD966")

for cell in ws[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    cell.border = border

headers = {cell.value: cell.column for cell in ws[1]}
numero_col = headers.get("numero_archivo")
id_etiqueta_col = headers.get("id_etiqueta")

for row_idx in range(2, ws.max_row + 1):
    numero = ws.cell(row=row_idx, column=numero_col).value

    try:
        numero_int = int(numero)
    except Exception:
        numero_int = 0

    fill = fill_odd if numero_int % 2 == 1 else fill_even
    fill_group = fill_group_odd if numero_int % 2 == 1 else fill_group_even

    for col_idx in range(1, ws.max_column + 1):
        cell = ws.cell(row=row_idx, column=col_idx)
        cell.fill = fill
        cell.border = border
        cell.alignment = Alignment(vertical="top", wrap_text=True)

    ws.cell(row=row_idx, column=numero_col).fill = fill_group
    ws.cell(row=row_idx, column=numero_col).font = Font(bold=True)

    if id_etiqueta_col:
        ws.cell(row=row_idx, column=id_etiqueta_col).font = Font(bold=True)

widths = {
    "numero_archivo": 16,
    "id": 14,
    "nombre_archivo": 26,
    "clasificacion": 18,
    "texto_limpio": 80,
    "id_etiqueta": 24,
    "etiqueta": 14,
    "id_variante": 14,
    "valor": 38,
    "metodo": 24,
    "span_inicio": 14,
    "span_fin": 14,
}

for header, width in widths.items():
    if header in headers:
        ws.column_dimensions[get_column_letter(headers[header])].width = width

ws.freeze_panes = "A2"
ws.auto_filter.ref = ws.dimensions

# Resumen
ws_resumen = wb.create_sheet("Resumen")
ws_resumen.append(["Métrica", "Valor"])
ws_resumen.append(["Documentos", len(json_personas_agrupadas)])
ws_resumen.append(["Total grupos persona", df_personas_agrupadas["id_etiqueta"].nunique()])
ws_resumen.append(["Total variantes persona", len(df_personas_agrupadas)])

ws_resumen.append([])
ws_resumen.append(["Variantes por método", "Cantidad"])

for metodo, cantidad in df_personas_agrupadas["metodo"].value_counts().items():
    ws_resumen.append([metodo, int(cantidad)])

for cell in ws_resumen[1]:
    cell.fill = header_fill
    cell.font = header_font

ws_resumen.column_dimensions["A"].width = 32
ws_resumen.column_dimensions["B"].width = 18

wb.save(OUTPUT_EXCEL)

print("Excel guardado:", OUTPUT_EXCEL)

Excel guardado: /content/embargos_personas_agrupadas_variantes.xlsx


# Celda 14 — Validaciones

In [15]:
print("Documentos:", len(json_personas_agrupadas))
print("Total grupos persona:", df_personas_agrupadas["id_etiqueta"].nunique())
print("Total variantes persona:", len(df_personas_agrupadas))

print("\nPromedio de grupos persona por archivo:")
print(
    df_personas_agrupadas.groupby("numero_archivo")["id_etiqueta"].nunique().mean()
)

print("\nPromedio de variantes por grupo persona:")
print(
    df_personas_agrupadas.groupby("id_etiqueta").size().mean()
)

print("\nTop 10 grupos con más variantes:")
print(
    df_personas_agrupadas.groupby(["numero_archivo", "id_etiqueta"])
    .size()
    .sort_values(ascending=False)
    .head(10)
)

Documentos: 80
Total grupos persona: 779
Total variantes persona: 1031

Promedio de grupos persona por archivo:
9.7375

Promedio de variantes por grupo persona:
1.3234916559691912

Top 10 grupos con más variantes:
numero_archivo  id_etiqueta    
27              027_persona_001    5
72              072_persona_002    5
40              040_persona_005    5
32              032_persona_001    5
75              075_persona_002    5
28              028_persona_001    4
10              010_persona_002    4
4               004_persona_002    4
11              011_persona_003    4
67              067_persona_001    4
dtype: int64


# Celda 15 — Descargar archivos

In [16]:
from google.colab import files

files.download(str(OUTPUT_JSON))
files.download(str(OUTPUT_CSV))
files.download(str(OUTPUT_EXCEL))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>